# Week 3 · Custom Behavior: Fine-Tuning SLMs with QLoRA
### Build Custom AI — SarasAI Live Session 3

Weeks 1–2 changed **what the model knows** (prompts, retrieval). This week we change
**how the model behaves** — we train the capstone's *Analyst*: a Small Language Model that
takes a defect note (Week 1) + retrieved evidence (Week 2) and writes a structured root-cause
report, every time, in exactly our format.

**The decision framework first** — when do you fine-tune instead of RAG?

| You need to change… | Use |
|---|---|
| Facts, freshness, citations | **RAG** (no training) |
| Format, tone, style, task-specific skill | **Fine-tuning** |
| Both | **Both** — they compose (this capstone!) |

**What you will do today**

1. Load a 1.5B SLM in **4-bit** and understand why that fits a free T4.
2. Build a **synthetic instruction dataset** with real data-quality discipline (dedup + leakage checks).
3. Configure **LoRA** and train with TRL's `SFTTrainer`.
4. Evaluate on a **held-out test set**: format-adherence, rubric score, win-rate vs. the base model.
5. Preview **DPO** — preference tuning without a reward model.

Primary sources: LoRA [arXiv:2106.09685](https://arxiv.org/abs/2106.09685) ·
QLoRA [arXiv:2305.14314](https://arxiv.org/abs/2305.14314) ·
DPO [arXiv:2305.18290](https://arxiv.org/abs/2305.18290)

> ✍️ **LIVE-CODING WORKBOOK** — same notebook as `week3_finetuning_analyst_qlora.ipynb`, with the teaching-core cells left as `# TODO (live)` skeletons we write together in the session. Boilerplate (installs, data generator, trainer config, save cells) is pre-filled. The fully-coded notebook is the answer key — keep it open next to this one.

---
## 0 · Setup

The fine-tuning stack: `peft` (LoRA adapters), `trl` (trainers), `bitsandbytes` (4-bit
quantization). All three are Hugging Face-native and compose out of the box.

In [ ]:
!pip install -q "transformers>=4.50" "trl>=0.17" "peft>=0.14" bitsandbytes datasets accelerate sentence-transformers

In [ ]:
import torch

assert torch.cuda.is_available(), "Fine-tuning needs a GPU — enable one in Runtime settings."
print("Device:", torch.cuda.get_device_name(0))

# T4s (pre-Ampere) have no native bfloat16 — detect once, use everywhere
BF16_OK = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print("Using dtype:", DTYPE)

---
## 1 · QLoRA in one picture

Full fine-tuning of even a 1.5B model needs ~24 GB (weights + gradients + optimizer states).
QLoRA gets that under 6 GB with two moves:

```
frozen base model, quantized to 4-bit   ←  QLoRA's "Q" (NF4 quantization)
        +
tiny trainable LoRA adapters (~0.5–1% of params)  ←  LoRA's low-rank matrices
```

- **LoRA** (Hu 2021): don't update the weight matrix `W`; learn a low-rank correction
  `ΔW = B·A` where `A`,`B` are thin matrices (rank r ≈ 8–64). Train ~1% of parameters,
  get ~full-fine-tune quality for behavior tasks.
- **QLoRA** (Dettmers 2023): keep the frozen base in **4-bit NF4** — gradients flow *through*
  the quantized weights *into* the adapters. Adds double-quantization and paged optimizers
  so nothing OOMs.

Let's load the base model exactly that way:

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# TODO (live): the QLoRA loading recipe — a BitsAndBytesConfig with the three tricks:
#   load_in_4bit=True · bnb_4bit_quant_type="nf4" · bnb_4bit_compute_dtype=DTYPE
#   · bnb_4bit_use_double_quant=True
# then AutoTokenizer + AutoModelForCausalLM(quantization_config=..., device_map="auto")
# and print get_memory_footprint()/1e9 — the number that makes training on a T4 possible
...

~1.2 GB for a 1.5B model — the same model in fp32 would be ~6 GB. That headroom is what
lets us *train* on a T4.

---
## 2 · Baseline: what does the base model do with our task?

Before training anything, capture the *before*. The Analyst's job — turn evidence into a
fixed-format report:

In [ ]:
REPORT_FORMAT = """## ROOT CAUSE ANALYSIS
**Product:** <model id>
**Defect:** <one-line summary>
**Severity:** LOW | MEDIUM | HIGH
**Evidence:** <cited facts from the input>
**Probable cause:** <one short paragraph>
**Recommended action:** <one concrete step>"""

def make_input(defect_note, evidence):
    return (f"Defect note from visual inspection:\n{defect_note}\n\n"
            f"Retrieved evidence:\n{evidence}\n\n"
            f"Write a root cause analysis report.")

example_input = make_input(
    "Bottle cap cracked near seal, hairline fracture visible on 3 units from batch B-1122.",
    "- review_041: 'lid cracked after one week of normal use'\n"
    "- spec_kettle: lid assembly is polypropylene, part PP-114, rated -10°C to 90°C\n"
    "- review_087: 'cap arrived already cracked in the box'",
)

In [ ]:
def chat(model, user_text, max_new_tokens=350):
    """One greedy chat turn — third time this course; write it from memory."""
    # TODO (live): apply_chat_template (add_generation_prompt, return_dict, "pt")
    #              -> generate (do_sample=False, pad_token_id=eos)
    #              -> decode ONLY the new tokens (slice off the prompt), strip
    ...

Typical result: *reasonable content, wrong shape* — headers missing or renamed, severity
free-texted, sections reordered. For a human reader that's fine; for a **pipeline** that parses
these reports downstream, it's broken. Format consistency is precisely what fine-tuning buys.

---
## 3 · The training data: synthetic generation with discipline

We have no labeled reports — so we synthesize them. This is the standard bootstrap
(Self-Instruct, arXiv:2212.10560): a *teacher* writes exemplary outputs, the student trains
on them. In production you'd use a frontier model as teacher; here we ship a curated seed set
inline so the session doesn't depend on API keys.

**The three data-quality rules that decide whether Week 3 works:**
1. **Diversity** — vary product, defect type, severity, evidence mix.
2. **Dedup** — near-duplicate training rows teach the model to memorize, not generalize.
3. **Leakage control** — nothing in the training set may resemble the held-out test set.

In [ ]:
import itertools, random

random.seed(42)

products = ["KT-2000 kettle", "TS-400 toaster", "BL-900 blender", "MW-70 microwave", "VF-12 ventilation fan"]
defects  = [("cracked lid hinge", "HIGH"), ("discolored housing", "LOW"),
            ("lever sticking", "MEDIUM"), ("seal degradation", "MEDIUM"),
            ("blade wobble", "HIGH"), ("loose power cord", "HIGH"),
            ("faded print", "LOW"), ("rattling component", "MEDIUM")]
causes   = ["material fatigue under thermal cycling", "supplier lot variation",
            "assembly torque out of spec", "UV degradation of the polymer",
            "shipping vibration", "worn tooling on the production line"]

def synth_example(product, defect, severity, cause, idx):
    note = f"{defect.capitalize()} observed on {random.randint(2,9)} units of {product}, batch B-{1000+idx}."
    evidence = (f"- review_{idx:03d}: customer reports '{defect}' within weeks of purchase\n"
                f"- spec: component rated for standard household use\n"
                f"- service log: similar reports {'rising' if severity != 'LOW' else 'stable'} this quarter")
    report = (f"## ROOT CAUSE ANALYSIS\n**Product:** {product}\n"
              f"**Defect:** {defect}\n**Severity:** {severity}\n"
              f"**Evidence:** Customer review corroborates inspection finding; service logs show "
              f"{'a rising trend' if severity != 'LOW' else 'stable reports'}.\n"
              f"**Probable cause:** {cause.capitalize()}.\n"
              f"**Recommended action:** "
              f"{'Quarantine the batch and audit the supplier lot.' if severity == 'HIGH' else 'Monitor the trend and schedule a design review.'}")
    return {"input": make_input(note, evidence), "output": report}

combos = list(itertools.product(products, defects, causes))
random.shuffle(combos)
raw_rows = [synth_example(p, d, s, c, i) for i, (p, (d, s), c) in enumerate(combos[:140])]
print(f"Generated {len(raw_rows)} candidate examples")

Rule 2 — **dedup**. We embed every input and drop rows too similar to an earlier one.
(Templated generators like ours produce *lots* of near-dupes — watch the count drop.)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
SIM_THRESHOLD = 0.90        # one threshold for BOTH hygiene checks

X = embedder.encode([r["input"] for r in raw_rows], normalize_embeddings=True)

# TODO (live): embedding DEDUP — keep a row only if its max cosine vs already-kept
# rows is <= SIM_THRESHOLD (np.dot on normalized vectors IS cosine). Print before/after.
...

Rule 3 — **split, then decontaminate**. We hold out a test set *now*, then actively **drop**
any train row that sits too close to a test row. Decontamination is an action, not a printout —
the final max-similarity number is the certificate that the split is clean.

> ⚠️ *Honest caveat:* our test rows come from the **same template generator** as training, so
> they're distributionally easy and metrics will look flattering. Your capstone data (a real
> teacher model, real variety) won't be this kind — which is exactly why the discipline matters.

In [ ]:
random.shuffle(keep)
test_rows, train_rows = keep[:12], keep[12:]

# TODO (live): DECONTAMINATE — an action, not a printout:
#   1. embed both sides -> sims = X_train @ X_test.T
#   2. DROP every train row whose max similarity to any test row exceeds SIM_THRESHOLD
#   3. print dropped count + the leakage certificate (max cross-similarity)
...

---
## 4 · LoRA configuration — the four numbers that matter

```python
r=16              # adapter rank: capacity of the correction. 8–32 typical for format tasks
lora_alpha=32     # scaling; rule of thumb alpha ≈ 2·r
target_modules    # which layers get adapters — all attention + MLP projections is the modern default
lora_dropout=0.05 # light regularization
```

Bigger `r` = more capacity = more risk of memorizing your (small) dataset. For a *format and
structure* task like ours, small ranks work well.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# TODO (live): the four LoRA numbers that matter —
#   r=16 · lora_alpha=32 (rule of thumb: 2r) ·
#   target_modules = attention + MLP projections for Qwen2.5:
#     q_proj k_proj v_proj o_proj gate_proj up_proj down_proj
#   lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
# then: model = get_peft_model(prepare_model_for_kbit_training(base), cfg)
#       model.print_trainable_parameters()   <- expect ~1-2%
...

Read that line carefully: **we train well under 1% of the parameters.** The adapter file
will be a few tens of MB — that's the whole artifact you ship, version, and merge later.

---
## 5 · Format the data & train

TRL's `SFTTrainer` accepts chat-format rows (`messages`) and applies the model's chat template
for us. One honest note on **loss masking**: ideally you'd train only on the assistant's tokens
(learn to *write reports*, not *write prompts*). TRL supports that (`assistant_only_loss=True`),
but it requires the chat template to carry `{% generation %}` markers — Qwen2.5's doesn't, so
here we train on the full sequence. With short prompts and long completions the difference is
small; for the capstone you can switch to prompt/completion format with `completion_only_loss`:

In [ ]:
from datasets import Dataset

def to_chat(row):
    return {"messages": [
        {"role": "user", "content": row["input"]},
        {"role": "assistant", "content": row["output"]},
    ]}

train_ds = Dataset.from_list([to_chat(r) for r in train_rows])
print(train_ds)

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_cfg = SFTConfig(
    output_dir="analyst-qlora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,     # effective batch = 8
    learning_rate=2e-4,                # LoRA tolerates higher LR than full FT
    lr_scheduler_type="cosine",
    logging_steps=5,
    bf16=BF16_OK, fp16=not BF16_OK,    # match the GPU's native half precision
    optim="paged_adamw_8bit",          # QLoRA trick #3: paged optimizer — no OOM spikes
    max_length=1024,
    report_to="none",
)

trainer = SFTTrainer(model=model, args=sft_cfg, train_dataset=train_ds,
                     processing_class=tok)
trainer.train()

A few minutes on a T4 (small dataset → few optimizer steps; your bigger capstone dataset
will take proportionally longer). While it runs, the loss curve narrative: a fast drop in the
first epoch (learning the *format*), then slow gains (learning the *phrasing*). If loss goes
to ~0, you've memorized — smaller `r`, fewer epochs, or more data.

---
## 6 · Before / after — then the *real* evaluation

The demo moment first:

In [ ]:
tuned_output = chat(model, example_input)
print("=== BASE MODEL ===\n", baseline_output[:400])
print("\n=== FINE-TUNED ===\n", tuned_output[:400])

Satisfying — but the Evaluation Rule says a side-by-side anecdote proves nothing.
Three measured numbers on the **frozen test set**:

1. **Format adherence** — does the output contain all six required sections? (Deterministic,
   cheap, zero LLM calls — always find the *mechanical* metric first.)
2. **Rubric score** — LLM-as-judge grades content quality 1–5 against the reference.
3. **Win-rate** — judge picks blind between base and tuned outputs.

In [ ]:
import re

REQUIRED = ["## ROOT CAUSE ANALYSIS", "**Product:**", "**Defect:**", "**Severity:**",
            "**Evidence:**", "**Probable cause:**", "**Recommended action:**"]

def format_ok(text):
    return all(h in text for h in REQUIRED) and bool(
        re.search(r"\*\*Severity:\*\*\s*(LOW|MEDIUM|HIGH)", text))

def format_rate(outputs):
    return sum(map(format_ok, outputs)) / len(outputs)

model.eval()   # after training: re-enables the KV cache (fast generation) and freezes dropout
test_inputs = [r["input"] for r in test_rows]

> ⚠️ **A trap worth knowing:** after `get_peft_model(base, …)`, the adapters are injected
> *into* `base`'s own module tree — calling `chat(base, …)` returns **tuned** outputs, always.
> There is no independent "base" object anymore. The correct way to get true base behavior is
> the `disable_adapter()` context manager:

In [ ]:
# TODO (live): the honest A/B — remember the trap: adapters are injected INTO base's
# module tree, so chat(base, ...) returns TUNED outputs. The clean way:
#   with model.disable_adapter():  ->  base_outputs
#   then tuned_outputs normally, and compare format_rate for both
...

Metric 2 — the **rubric score**: an LLM judge grades each report 1–5 against the reference
output. (Same honesty rules as Week 2: for the capstone use a *stronger, different* judge —
this in-family judge is a smoke test of the *protocol*.)

In [ ]:
def rubric_score(inp, generated, reference):
    """Judge grades 1-5: format, groundedness in the input, agreement with the reference."""
    prompt = (f"Reference report:\n{reference[:600]}\n\nCandidate report:\n{generated[:600]}\n\n"
              "Grade the candidate 1-5 (5 = matches the reference's facts and format, fully "
              "grounded; 1 = wrong format or invented facts). Reply with ONLY the digit.")
    with model.disable_adapter():
        verdict = chat(model, prompt, max_new_tokens=3)
    m = re.search(r"[1-5]", verdict)
    return int(m.group()) if m else 1

scores_base  = [rubric_score(r["input"], o, r["output"]) for r, o in zip(test_rows, base_outputs)]
scores_tuned = [rubric_score(r["input"], o, r["output"]) for r, o in zip(test_rows, tuned_outputs)]
print(f"Rubric (mean 1-5) — base: {sum(scores_base)/len(scores_base):.2f}   "
      f"tuned: {sum(scores_tuned)/len(scores_tuned):.2f}")

In [ ]:
def judge_winner(inp, a, b):
    """Blind pairwise judge: which report is better? Returns "A" or "B"."""
    # TODO (live): judge prompt over (input, Report A, Report B) -> one letter.
    # Then the win-rate loop — RANDOMIZE which side is tuned (position bias is real),
    # track where the tuned output sat, count a win when the judge picked that slot.
    ...

> Same caveat as Week 2: a 1.5B model judging its own family is a smoke test. For the
> capstone submission, use a stronger, *different* judge and randomize A/B order. The judging
> **pattern** is what you're learning here.

Save the adapter — note the size. *This* is your deliverable artifact:

In [ ]:
import json, os

model.save_pretrained("analyst-lora-adapter")

# Week 4 re-scores THIS frozen test set after compression — persist it now
with open("week3_test_rows.json", "w") as f:
    json.dump(test_rows, f, indent=1)

size = sum(os.path.getsize(os.path.join("analyst-lora-adapter", f))
           for f in os.listdir("analyst-lora-adapter")) / 1e6
print(f"Adapter size: {size:.0f} MB   (the 1.5B base stays untouched on the Hub)")
print("Frozen test set saved → week3_test_rows.json  (Week 4 needs both artifacts)")
# to publish: model.push_to_hub("your-username/analyst-lora-adapter")

---
## 7 · Preview: DPO — teaching *preferences*, not examples

SFT teaches "here is a good output — imitate it." But some qualities are easier to show as
**comparisons**: "report A is better than report B" (more concise, better calibrated severity,
no invented facts). That's preference alignment.

Classic RLHF needs a separate reward model + RL loop. **DPO** (Rafailov 2023,
arXiv:2305.18290) collapses all of it into a simple classification-style loss on
(prompt, chosen, rejected) triplets — no reward model, no RL. In TRL it's a drop-in trainer:

```python
from trl import DPOConfig, DPOTrainer

pref_ds = Dataset.from_list([
    {"prompt": "...", "chosen": "<grounded, concise report>",
     "rejected": "<verbose report that invents an unsupported cause>"},
    # a few hundred pairs is often enough on top of an SFT'd model
])
trainer = DPOTrainer(model=model, args=DPOConfig(output_dir="analyst-dpo", beta=0.1),
                     train_dataset=pref_ds, processing_class=tok)
```

Typical recipe: **SFT first (today), DPO second** to sand off the failure modes your eval
surfaces. If your Increment-3 eval shows e.g. severity over-calling, a small DPO pass on
chosen/rejected severity pairs is the stretch goal.

---
## 8 · Production considerations: data quality & eval discipline

For fine-tuning, **the dataset is the product** — the training loop is commodity.

| Notebook habit | Production discipline |
|---|---|
| Generate once, train once | **Version datasets** like code (hashes, changelogs) |
| Eyeball a few rows | Dedup + leakage checks **in CI**, gating every training run |
| "The new run feels better" | Frozen eval re-run on **every** change; deltas tracked |
| One big noisy dataset | **Small and clean beats big and dirty** — curation is the job |

And the one that bites hardest: **synthetic data amplifies leakage risk.** A teacher model
paraphrasing the same seed produces train/test twins that string-matching won't catch — that's
why our leakage check used embeddings, not exact matches.

---
## 9 · Your assignments

**Ungraded warm-up:** regenerate the synthetic dataset with a real teacher (any strong chat
model), 100+ examples. Run the dedup + leakage cells unchanged. Report what fraction survived.

**Graded — Capstone Increment 3:** QLoRA-fine-tune the Analyst on your capstone data
(defect notes from Increment 1 + retrieved evidence from Increment 2 as inputs). Submit:
(1) adapter weights or Hub link, (2) dataset card — sizes, dedup rate, max train↔test similarity,
(3) held-out results: format-adherence, rubric/judge protocol + scores, win-rate vs. base.

**Read before next week** *(≈40 min)*:
- Distillation — Hinton et al. 2015, [arXiv:1503.02531](https://arxiv.org/abs/1503.02531) (§1–2)
- GPTQ [arXiv:2210.17323](https://arxiv.org/abs/2210.17323) *or* AWQ
  [arXiv:2306.00978](https://arxiv.org/abs/2306.00978) (abstract + intro)

*Next week: your Analyst works. We make it small, fast, cheap — and ship the whole pipeline.*